<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c41-header.png" alt="Nextia Learning · Speech AI: Transcribe, Understand and Speak" width="100%">

# Menu, agent or both?

**[Menu, agent or both?](https://learning.nextia-ai.com/courses/speech/m08/menu-agent-or-both/)** · Speech AI: Transcribe, Understand and Speak · Module 8, lesson 5 of 5 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** compare three designs of a phone line on the same recorded test calls: the IVR, the voice agent and a hybrid. You compute containment, hand-overs, wrong actions and the cost per call, find what the cost leaves out, and test how the choice changes when an assumption changes.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key: the agent's answers are recorded. |
| **You should know** | How to evaluate a voice agent as a whole, from [Speech-to-speech models and evaluation](https://learning.nextia-ai.com/courses/speech/m08/speech-to-speech-models-and-evaluation/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, SciPy 1.18.1, soundfile 0.14.0 and matplotlib 3.11.2 (scikit-learn 1.9.1, jiwer 4.0.0 and faster-whisper 1.2.1 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/speech/m08/menu-agent-or-both/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M08-L05-menu-agent-or-both/menu-agent-or-both-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, scipy 1.18.1, soundfile 0.14.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 scipy==1.18.1 soundfile==0.14.0 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'scipy', 'soundfile']:
    print(name, version(name))

## Setup: get the Larkfield calls

Run the next cell. It downloads the files that this lesson needs (about 1 MB) into a folder `larkfield-calls`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-calls` (in Colab: the **Files** panel on the left), then run the cell again.

Where the calls come from: Larkfield is an invented shop. Our code wrote the scripts, and open text-to-speech voices (Piper and Kokoro) read them. Every call is **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/data/dataset-card.md) names each voice and its licence. The calls are free to reuse under CC BY 4.0.

In [ ]:
# Setup: download the files into larkfield-calls/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/"
FILES = {
    "data/recordings/agent/ivr_replay.jsonl": "d4d8dd6de71037e7ab6c2ca0433214ec3cfec5112a2de42283c4933994372438",
    "data/recordings/agent/calls.jsonl": "55157881d8bd2cd2738aaa4591eb87e292c790fca98896425bb73580f8b97f38",
    "data/costs.json": "71133d6f2b813a316d00b7aa965e8dfe66037c7936ef6b2e298ebe2578f8174a",
    "compare.py": "db394eb044e0423706644ba7c8e6451e4d21a268293b868032918ea2f30db85b",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-calls").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-calls and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-calls.")

## Setup: helpers

The next cell loads NumPy as `np`, soundfile as `sf`, SciPy's `signal`, matplotlib, the audio player `Audio`, and the course helper as `cp` (normalisation, word error rate and the comparison table). It defines a few small helpers:

- `load_calls()` reads the labels of every call; `reference(call)` joins its turns into the reference transcript.
- `read_audio(path, sr, channel)` reads a recording as numbers between -1 and 1; `dbfs(x)` gives its loudness.
- `true_turns(call)` gives the true (start, end, speaker) of each turn, from the first word to the last word.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: calls, audio, and checks that do not show the answer.
import hashlib, json, warnings
from math import gcd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
import compare as cp
warnings.filterwarnings("ignore")

def load_calls(path="data/calls/calls.jsonl"):
    """The labels of every call, by call ID."""
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    return {c["call_id"]: c for c in rows}

def reference(call, speaker=None):
    """The reference transcript: the text of all turns (or of one speaker's turns), in time order."""
    return " ".join(t["text"] for t in call["turns"] if speaker in (None, t["speaker"]))

def read_audio(path, sr=None, channel=None):
    """(numbers between -1 and 1, samples a second). channel=None mixes the channels; 0 or 1 keeps one side."""
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    x = x.mean(axis=1) if channel is None else x[:, channel]
    if sr and sr != file_sr:
        g = gcd(file_sr, sr)
        x = signal.resample_poly(x, sr // g, file_sr // g).astype(np.float32)
    return x, sr or file_sr

def dbfs(x):
    """Loudness in decibels below full scale: 0 is the loudest; speech on a phone is often -30 to -15."""
    return float(20 * np.log10(np.sqrt(np.mean(np.square(x))) + 1e-9))

def true_turns(call):
    """The true turns: (start, end, speaker), from the first word to the last word of each turn."""
    return [(t["words"][0]["start"], t["words"][-1]["end"], t["speaker"]) if t.get("words")
            else (t["start"], t["end"], t["speaker"]) for t in call["turns"]]

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Two recordings of the same calls

Omar must choose a design for Larkfield's out-of-hours line. The course team replayed the 60 IVR test calls twice: once through the IVR, once through the voice agent (Azure OpenAI `chat-small` with tools, recorded). Each line of the files is one call. Run the next cell.

In [ ]:
def read_jsonl(path):
    return [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]

ivr = {r["call_id"]: r for r in read_jsonl("data/recordings/agent/ivr_replay.jsonl")}
agent = {r["call_id"]: r for r in read_jsonl("data/recordings/agent/calls.jsonl")}
costs = json.load(open("data/costs.json"))
print(len(ivr), "IVR calls,", len(agent), "agent calls (the same test calls)")
print({k: costs[k] for k in ("call_handled_by_person", "wrong_action", "caller_hangs_up", "ivr_call_compute")})
r, a = ivr["I-001"], agent["I-001"]
print("I-001 IVR:  ", r["outcome"], r["handoff_reason"], r["seconds"], "s")
print("I-001 agent:", a["outcome"], a["seconds"], "s, US$", a["usd"])

> **Check.** 60 calls in each file. The costs are **assumptions** from the course's cost table: a call handled by a person costs US$4.00, a wrong action US$15, a caller who hangs up US$8. Test call I-001 went to a person in the IVR, and ended with no answer in the agent ("hangup": the recorded caller ran out of lines while the agent still asked).

## 2. Three designs

The **hybrid** puts the menu first. Only when the menu fails twice to match the caller's words (a hand-over with the reason `no_match`) does the call go to the agent. Every other call stays with the IVR. The next cell builds the three designs and prints the same table as the project command `speech agent compare`, with one more column: the calls with no end.

In [ ]:
CONTAINED = ("status_given", "hours_given", "return_started")

def designs(ivr, agent, ivr_compute):
    """Each design's result for each call. The hybrid: the IVR first; the agent only after an IVR no-match."""
    out = {"IVR (menu)": {}, "voice agent": {}, "hybrid": {}}
    for cid, r in ivr.items():
        a = agent[cid]
        out["IVR (menu)"][cid] = {"outcome": r["outcome"], "seconds": r["seconds"], "wrong": r["wrong_intent"],
                                  "usd": ivr_compute}
        out["voice agent"][cid] = {"outcome": a["outcome"], "seconds": a["seconds"], "wrong": a["wrong_action"],
                                   "usd": a["usd"]}
        if r["outcome"] == "handover" and r.get("handoff_reason") == "no_match":
            out["hybrid"][cid] = {"outcome": a["outcome"], "seconds": r["seconds"] + a["seconds"],
                                  "wrong": a["wrong_action"], "usd": a["usd"] + ivr_compute}
        else:
            out["hybrid"][cid] = out["IVR (menu)"][cid]
    return out

def table(rows, costs, hangup_cost=0.0):
    """The project's comparison, with an optional cost for each call that ends with no answer (a hang-up)."""
    n = len(rows)
    contained = sum(v["outcome"] in CONTAINED and not v["wrong"] for v in rows.values())
    handed = sum(v["outcome"] == "handover" for v in rows.values())
    wrong = sum(bool(v["wrong"]) for v in rows.values())
    hangups = sum(v["outcome"] == "hangup" for v in rows.values())
    usd = (sum(v["usd"] for v in rows.values()) + handed * costs["call_handled_by_person"]
           + wrong * costs["wrong_action"] + hangups * hangup_cost)
    return {"contained": contained / n, "to a person": handed / n, "wrong": wrong, "hangups": hangups,
            "seconds": sum(v["seconds"] for v in rows.values()) / n, "usd": usd / n}

D = designs(ivr, agent, costs["ivr_call_compute"])
print(f"{'design':12} {'contained':>10} {'to a person':>12} {'wrong':>6} {'no end':>7} {'mean s':>7} {'US$ a call':>11}")
for name, rows in D.items():
    t = table(rows, costs)
    print(f"{name:12} {t['contained']:>10.0%} {t['to a person']:>12.0%} {t['wrong']:>6} {t['hangups']:>7} "
          f"{t['seconds']:>7.1f} {t['usd']:>11.3f}")

> **Check.** The IVR contains 35% at US$3.035 a call; the agent costs US$2.8; the hybrid contains 37% at US$2.702. The hybrid looks best on both. But look at the column "no end": the agent leaves 7 calls with no answer, and the hybrid 4.

## 3. What does the agent do in the hybrid?

In [ ]:
from collections import Counter
no_match = [cid for cid, r in ivr.items() if r["outcome"] == "handover" and r.get("handoff_reason") == "no_match"]
print(len(no_match), "calls go to the agent in the hybrid; the agent's outcomes on them:")
print(Counter(agent[cid]["outcome"] for cid in no_match))

> **Check.** 16 calls go to the agent. Most of them still go to a person. Only one ends with an answer. Some end with no answer at all: these calls are no longer charged as hand-overs, so the hybrid **looks** cheaper.

## 4. Where the cost comes from

In [ ]:
for name, rows in D.items():
    n = len(rows)
    handed = sum(v["outcome"] == "handover" for v in rows.values())
    wrong = sum(bool(v["wrong"]) for v in rows.values())
    print(f"{name:12} people {handed * costs['call_handled_by_person'] / n:.3f} + wrong actions "
          f"{wrong * costs['wrong_action'] / n:.3f} + compute and LLM {sum(v['usd'] for v in rows.values()) / n:.4f}")

> **Check.** People are almost all of the cost. The LLM costs less than a tenth of a cent a call. A design is cheap when it sends fewer calls to a person, **and** when it does not leave callers with no answer.

## 5. Your turn: count the hang-ups

> **Your turn.** A call with no end leaves the caller without an answer. In a real call, the caller would hang up and call again. Count each such call at the course's cost of a hang-up: `costs["caller_hangs_up"]` (US$8, an assumption). Use `table(rows, costs, hangup_cost=...)` for each design. Put the cost per call of each design in a dictionary `with_hangups` (rounded to 3 decimals), and the name of the cheapest design in `cheapest`.

In [ ]:
with_hangups = ...
cheapest = ...

In [ ]:
expect("the cheapest design", cheapest, 'IVR (menu)')
expect_hash("with_hangups", with_hangups, 'ca92b0fd75e08f6b')

> **Check.** With the hang-ups counted, the IVR costs US$3.169 a call, the hybrid US$3.235 and the agent US$3.734. The cheapest design is now the **IVR (menu)**. One assumption changed the ranking: always show the costs that a table leaves out.

## 6. How sure is the choice?

The costs are assumptions. The next cell changes the cost of a person and prints the cost per call of each design (with the hang-ups).

In [ ]:
for person in (3.0, 4.0, 6.0):
    c = {**costs, "call_handled_by_person": person}
    row = {name: round(table(rows, c, costs["caller_hangs_up"])["usd"], 2) for name, rows in D.items()}
    print(f"a person costs US${person:.2f}: {row}")

> **Check.** At US$3 and US$4 a person, the IVR is the cheapest. At US$6, the hybrid is: it sends 5 fewer calls in 60 to a person, and that saving grows with the cost of a person. The two are equal at about US$4.80. The choice depends on an assumption, so say which one. And the hybrid contains only one more call than the IVR in 60: that is within noise. The hybrid needs a test with callers who can react before it can win.

## Next

You compared the IVR, the agent and a hybrid on the same recorded calls, found the cost that the table leaves out, and tested the choice against its assumptions. The [lesson page](https://learning.nextia-ai.com/courses/speech/m08/menu-agent-or-both/) has the recommendation for Larkfield and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Batch and streaming](https://learning.nextia-ai.com/courses/speech/m09/batch-and-streaming/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/speech/m08/menu-agent-or-both/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every call is synthetic: open text-to-speech voices read scripts that we invented. No real customer and no real person's voice is in this data.